In [1]:
import pandas as pd

videos_per_topic_docs_df = pd.read_csv("videos_per_topic_docs.csv")
video_id_to_doc_mapping_df = pd.read_csv("video_id_to_doc_mapping.csv")

video_id_to_doc_mapping_df


,Video Id,Video Title,Doc Name
0,--8n6A8Q6M0,$200 Luxury Beach Hotel in The Philippines 🇵🇭,0
1,-1B7cVoZr1c,Marine reacts to the Philippine Light Reaction...,1
2,-7vF5F-1btE,Ultimate Filipino Food Festival In The Netherl...,2
3,-9bfDHHneyU,SHOWING MY SISTER SB19 'GENTO' Music Video,3
4,-C5iB25BRsA,10 Reasons/Do not Retire TO the Philippines/Mo...,4
...,...,...,...
2699,zm_8N4vFnZw,PHILIPPINES PROVINCE LIFE IS SO DIFFERENT FROM...,2699
2700,zpT46etTP6E,"Starring real-life gay couple, latest Filipino...",2700
2701,zqvDHfgxWh8,🇰🇷Koreans React to P-pop Idol Group | Lovey Do...,2701
2702,zvTP6wl9sTU,FOREIGNER LIVING IN YHE PHILIPPINES WEEKLY FO...,2702


In [2]:
# videos_per_topic_docs_df = videos_per_topic_docs_df.iloc[1:].reset_index(drop=True)

videos_per_topic_docs_df

,Topic,Custom Topic Label,Video_Ids
0,-1,Topic -1: Outliers,"['178', '358', '495', '685', '704', '822', '89..."
1,0,Topic 0: Delicious Dishes and Dining,"['0', '2', '3', '4', '5', '7', '8', '9', '10',..."
2,1,Topic 1: Learning and Speaking Foreign Language,"['23', '34', '36', '40', '48', '59', '69', '10..."
3,2,Topic 2: Christmas Season & Celebration,"['5', '19', '23', '32', '35', '44', '47', '50'..."
4,3,Topic 3: Experiences of Children in Community ...,"['9', '22', '24', '70', '127', '131', '132', '..."
...,...,...,...
117,116,Topic 116: Chicken & its Specific Related Dishes,"['210', '263', '277', '323', '374', '380', '39..."
118,117,Topic 117: Filipino Phrases,"['18', '29', '84', '148', '282', '353', '407',..."
119,118,Topic 118: Mommy and Baby's Morning Events,"['27', '44', '76', '162', '244', '256', '268',..."
120,119,Topic 119: Common Filipino Dishes,"['26', '323', '367', '436', '599', '695', '766..."


In [3]:
import ast

# Convert the Video_Ids column from string to list
videos_per_topic_docs_df["Video_Ids"] = videos_per_topic_docs_df["Video_Ids"].apply(ast.literal_eval)


In [4]:
import json
import pandas as pd

# HLTM
topic_map_filepath = './HLTM/output-jsons/' + 'T3.topics.json'

with open(topic_map_filepath, 'r') as f:
    data = json.load(f)

topic_data = []
for entry in data:
    topic_id = entry['topic']
    video_ids = [doc[0] for doc in entry['doc']]  # discard probabilities
    topic_data.append({'topic_id': topic_id, 'video_ids': video_ids})

# Convert to DataFrame
hltm_df = pd.DataFrame(topic_data)

# Display the first few rows
hltm_df.head()




,topic_id,video_ids
0,Z22,"[65, 105, 321, 337, 494, 514, 536, 721, 759, 8..."
1,Z1174,"[5, 19, 21, 25, 41, 43, 52, 54, 71, 76, 86, 88..."
2,Z1197,"[1, 3, 8, 17, 27, 33, 41, 43, 51, 52, 54, 58, ..."
3,Z11,"[811, 2479, 538, 1908, 2643, 887, 1869, 2280, ..."
4,Z232,"[51, 144, 168, 191, 343, 660, 685, 732, 1054, ..."


In [5]:
from collections import defaultdict

# Build per-video topic sets
video_topics = defaultdict(set)

# Add BERTopic assignments (prefix with B_)
for _, row in videos_per_topic_docs_df.iterrows():
    topic_id = f"B_{row['Topic']}"
    for doc_id in map(int, row["Video_Ids"]):
        video_topics[doc_id].add(topic_id)

# Add HLTM assignments (prefix with H_)
for _, row in hltm_df.iterrows():
    topic_id = f"H_{row['topic_id']}"
    for doc_id in row["video_ids"]:
        video_topics[doc_id].add(topic_id)

# Convert to transaction DataFrame
transactions_df = pd.DataFrame([
    {"doc_id": doc_id, "topics": list(topics)}
    for doc_id, topics in video_topics.items()
])


In [6]:
# If transactions_df has multiple rows per doc_id, each with its own topics list
# Group them and merge all the topic lists per doc_id
transactions_df = (
    transactions_df
    .groupby("doc_id")["topics"]
    .agg(lambda lists: sorted(set(sum(lists, []))))  # flatten + deduplicate + sort
    .reset_index()
)


In [7]:
transactions_df

,doc_id,topics
0,0,"[B_0, B_10, B_43, B_59]"
1,1,"[B_25, B_4]"
2,2,"[B_0, B_65]"
3,3,[B_0]
4,4,"[B_0, B_30, B_58, B_64, B_9, B_94]"
...,...,...
5391,995,"[H_Z1101, H_Z1103, H_Z1104, H_Z1109, H_Z111, H..."
5392,996,"[H_Z1105, H_Z1108, H_Z111, H_Z1118, H_Z1120, H..."
5393,997,"[H_Z1103, H_Z1105, H_Z1107, H_Z1110, H_Z1141, ..."
5394,998,"[H_Z1121, H_Z165]"


In [ ]:
import pandas as pd

transactions_df["topics"] = transactions_df["topics"].apply(lambda x: eval(x) if isinstance(x, str) else x)

transactions_df["doc_id"] = pd.to_numeric(transactions_df["doc_id"], errors="coerce")

transactions_df = transactions_df.dropna(subset=["doc_id"])
transactions_df["doc_id"] = transactions_df["doc_id"].astype(int)

def merge_lists(group):
    all_topics = []
    for lst in group:
        if isinstance(lst, list):
            all_topics.extend(lst)
    return list(dict.fromkeys(all_topics)) 

merged_df = transactions_df.groupby("doc_id", as_index=False)["topics"].agg(merge_lists)

merged_df = merged_df.sort_values("doc_id").reset_index(drop=True)

print(merged_df.head())


   doc_id                                             topics
0       0  [B_0, B_10, B_43, B_59, H_Z110, H_Z1100, H_Z11...
1       1  [B_25, B_4, H_Z110, H_Z1105, H_Z115, H_Z1158, ...
2       2  [B_0, B_65, H_Z1114, H_Z1126, H_Z1127, H_Z1128...
3       3  [B_0, H_Z11, H_Z1107, H_Z115, H_Z1162, H_Z1163...
4       4  [B_0, B_30, B_58, B_64, B_9, B_94, H_Z1101, H_...


In [9]:
merged_df

,doc_id,topics
0,0,"[B_0, B_10, B_43, B_59, H_Z110, H_Z1100, H_Z11..."
1,1,"[B_25, B_4, H_Z110, H_Z1105, H_Z115, H_Z1158, ..."
2,2,"[B_0, B_65, H_Z1114, H_Z1126, H_Z1127, H_Z1128..."
3,3,"[B_0, H_Z11, H_Z1107, H_Z115, H_Z1162, H_Z1163..."
4,4,"[B_0, B_30, B_58, B_64, B_9, B_94, H_Z1101, H_..."
...,...,...
2699,2699,"[B_0, B_61, H_Z110, H_Z1101, H_Z1105, H_Z1108,..."
2700,2700,"[B_0, B_3, B_35, B_39, B_78, H_Z1168, H_Z130, ..."
2701,2701,"[B_0, B_35, B_49, H_Z1109, H_Z1152, H_Z1162, H..."
2702,2702,"[B_0, B_26, H_Z1100, H_Z1127, H_Z1158, H_Z1167..."


In [10]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# Assume your DataFrame is called df
# Step 1: Explode each transaction into one-hot encoded format
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
onehot = pd.DataFrame(mlb.fit_transform(merged_df['topics']), 
                      columns=mlb.classes_,
                      index=merged_df.index)


In [11]:
onehot_bool = onehot.astype(bool)

In [12]:
frequent_itemsets = apriori(onehot_bool, min_support=0.01, use_colnames=True)


: 

In [ ]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)


In [ ]:
# Keep only rules where antecedent and consequent are from different models
def is_cross_model(antecedents, consequents):
    return (any(a.startswith('B_') for a in antecedents) and any(c.startswith('H_') for c in consequents)) or \
           (any(a.startswith('H_') for a in antecedents) and any(c.startswith('B_') for c in consequents))

rules['is_cross_model'] = rules.apply(lambda row: is_cross_model(row['antecedents'], row['consequents']), axis=1)
cross_model_rules = rules[rules['is_cross_model']]


###

In [ ]:
transactions_df = pd.DataFrame([
    {"doc_id": doc_id, "topics": list(topics)}  # ✅ topics is a full list per doc_id
    for doc_id, topics in video_topics.items()
])

print(transactions_df[transactions_df["doc_id"] == 1])

In [ ]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_array = te.fit(transactions_df["topics"]).transform(transactions_df["topics"])
onehot_df = pd.DataFrame(te_array, columns=te.columns_)


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

# Find frequent topic combinations
frequent_itemsets = apriori(onehot_df, min_support=0.05, use_colnames=True)

# Extract rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

# Example view
rules[["antecedents", "consequents", "support", "confidence", "lift"]]


In [ ]:
merged_df.to_csv("transactions.csv", index=False)

In [ ]:
# Show all rows
pd.set_option("display.max_rows", None)

# Show all columns (optional)
pd.set_option("display.max_columns", None)

# Disable column width truncation (optional)
pd.set_option("display.max_colwidth", None)

# Now display the full DataFrame
rules
